# predictive_drawdown_risk_gate_v1

Notebook-owned diagnostic for the active objective: preserve strict-OOS excess above `+30pp`, push MaxDD below `15%`, and avoid overfitting by requiring saved walk-forward validation before promotion.

Governance:
- Canonical baseline: `risk_budget_residual_drawdown_guard_v1`, because it is the first refreshed strict daily target pass (`+30.46pp`, `14.64%` MaxDD) but is blocked by saved walk-forward validation.
- Direct comparison baseline: `risk_budget_crash_risk_factor_v1`, because it nearly passes strict risk (`15.48%` MaxDD) with better excess (`+44.60pp`) before the residual guard over-defense.
- Role: `diagnose`
- Changed layer: `model_selection` + `execution`
- Target window: strict OOS `2020-01-02..2026-05-21` plus saved `wf_2020`, `wf_2022`, `wf_2023_2024` actor paths.
- Accepted trade-off: strict pass may lose some return versus crash-risk if every saved walk-forward window remains no worse than the residual guard risk path and does not lose the 2022 defense.
- Hypothesis: a purged expanding-window supervised drawdown-risk classifier can decide when to keep the residual guard active and when to release back to the crash-risk path, improving `wf_2020` underparticipation without losing strict risk control.
- Failure mode: if the classifier either reverts to residual guard most days or releases too much risk around the drawdown event, then the bottleneck is not an overlay classifier; the next step should move into reward/objective or actor architecture.
- Comparison-table row key: `predictive_drawdown_risk_gate_v1`

External basis:
- Deep Momentum Networks: https://arxiv.org/abs/1904.04912 (learn position sizing jointly instead of relying only on hand-tuned sizing rules)
- Constrained Max Drawdown: https://arxiv.org/abs/2401.02601 (drawdown as a direct optimization constraint)
- Qlib GitHub: https://github.com/microsoft/qlib (keep research artifacts replayable inside the qlib-style research-to-production harness)
- Expected Maximum Drawdown RRL: https://doi.org/10.1016/j.eswa.2017.06.023 (drawdown-aware learning objective motivation)

Historical reflection:
- Closest success: `risk_budget_residual_drawdown_guard_v1` strict pass, but only before saved walk-forward validation.
- Closest failure: `risk_budget_residual_drawdown_guard_walkforward_v1`, where residual guard cut risk but failed `wf_2020` and `wf_2023_2024`.
- Why this is not a duplicate: this is not a tighter cash cap. It trains a purged, expanding-window probability model from pre-window features and uses it to choose between two existing fixed actor paths: crash-risk participation versus residual-guard defense.


In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('/Users/rziiiii/.openclaw/workspace')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ['PYTHONPATH'] = 'src:.'

CARD = 'predictive_drawdown_risk_gate_v1'
CARD_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / CARD
CARD_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
STRICT_START = pd.Timestamp('2020-01-02')
STRICT_END = pd.Timestamp('2026-05-21')

PANEL_PATH = ROOT / 'data' / 'processed' / 'panels' / 'soxl_soxx_vix_full.csv'
STRICT_CRASH_PATH = ROOT / 'runtime' / 'execution' / 'card' / 'risk_budget_crash_risk_factor_v1' / 'risk_budget_crash_risk_factor_v1_selected_soxl_daily.csv'
STRICT_RESIDUAL_PATH = ROOT / 'runtime' / 'execution' / 'card' / 'risk_budget_residual_drawdown_guard_v1' / 'risk_budget_residual_drawdown_guard_v1_selected_soxl_daily.csv'
WF_DIR = ROOT / 'runtime' / 'execution' / 'card' / 'risk_budget_residual_drawdown_guard_walkforward_v1' / 'walkforward'
WF_WINDOWS = ['wf_2020', 'wf_2022', 'wf_2023_2024']

pd.set_option('display.max_columns', 120)
print('card_dir', CARD_DIR.relative_to(ROOT))
print('panel', PANEL_PATH.relative_to(ROOT))


card_dir runtime/model_selection/card/predictive_drawdown_risk_gate_v1
panel data/processed/panels/soxl_soxx_vix_full.csv


In [2]:
def build_feature_frame(panel: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    df = panel.copy()
    df['datetime'] = pd.to_datetime(df['date'])
    df = df.sort_values('datetime').reset_index(drop=True)

    soxl = df['soxl_close'].astype(float)
    soxx = df['soxx_close'].astype(float)
    vix = df['vix'].astype(float)
    soxl_ret = soxl.pct_change()
    soxx_ret = soxx.pct_change()
    ratio = soxl / soxx

    features = pd.DataFrame({'datetime': df['datetime']})
    for window in [5, 10, 21, 42, 63, 126, 252]:
        min_periods = max(3, window // 2)
        features[f'mom_{window}'] = soxl / soxl.shift(window) - 1.0
        features[f'drawdown_{window}'] = soxl / soxl.rolling(window, min_periods=min_periods).max() - 1.0
        features[f'rv_{window}'] = soxl_ret.rolling(window, min_periods=min_periods).std() * np.sqrt(252)
        features[f'downside_vol_{window}'] = soxl_ret.where(soxl_ret < 0.0, 0.0).rolling(window, min_periods=min_periods).std() * np.sqrt(252)

    for window in [50, 100, 200]:
        features[f'ma{window}_gap'] = soxl / soxl.rolling(window, min_periods=window // 2).mean() - 1.0

    features['soxx_mom_21'] = soxx / soxx.shift(21) - 1.0
    features['soxx_mom_63'] = soxx / soxx.shift(63) - 1.0
    features['soxl_soxx_ratio_widen_21'] = ratio / ratio.shift(21) - 1.0
    features['soxl_soxx_ratio_z_252'] = (ratio - ratio.rolling(252, min_periods=126).mean()) / ratio.rolling(252, min_periods=126).std()
    features['tracking_error_21'] = soxl_ret.rolling(21, min_periods=10).sum() - 3.0 * soxx_ret.rolling(21, min_periods=10).sum()
    features['vix_pct_252'] = vix.rolling(252, min_periods=126).rank(pct=True)
    features['vix_z_126'] = (vix - vix.rolling(126, min_periods=63).mean()) / vix.rolling(126, min_periods=63).std()
    features['vix_chg_5'] = vix / vix.shift(5) - 1.0
    features['vol_of_vol_21'] = features['rv_5'].rolling(21, min_periods=10).std()

    feature_cols = [col for col in features.columns if col != 'datetime']
    features[feature_cols] = features[feature_cols].shift(1)  # decision uses previous close features only
    features['soxl_close'] = soxl.to_numpy(float)
    return features, feature_cols


def add_forward_drawdown_label(features: pd.DataFrame, horizon: int, loss_threshold: float) -> pd.DataFrame:
    out = features.copy()
    close = out['soxl_close']
    forward_low = pd.concat([close.shift(-step) for step in range(1, horizon + 1)], axis=1).min(axis=1)
    out['future_min_return'] = forward_low / close - 1.0
    out['drawdown_label'] = (out['future_min_return'] <= -loss_threshold).astype(int)
    return out


def compute_metrics(frame: pd.DataFrame) -> dict:
    prices = frame['price'].to_numpy(float)
    positions = frame['position'].to_numpy(float)
    returns = np.diff(prices) / prices[:-1]
    strategy_returns = positions[:-1] * returns
    nav = np.r_[1.0, np.cumprod(1.0 + strategy_returns)]
    buyhold_nav = np.r_[1.0, np.cumprod(1.0 + returns)]
    drawdown = nav / np.maximum.accumulate(nav) - 1.0
    sharpe = float(np.mean(strategy_returns) / np.std(strategy_returns) * np.sqrt(252)) if len(strategy_returns) > 1 and np.std(strategy_returns) > 0 else 0.0
    return {
        'return': float((nav[-1] - 1.0) * 100.0),
        'buyhold_return': float((buyhold_nav[-1] - 1.0) * 100.0),
        'relative_return': float((nav[-1] - buyhold_nav[-1]) * 100.0),
        'sharpe': sharpe,
        'max_dd': float(abs(drawdown.min() * 100.0)),
        'avg_pos': float(np.mean(positions)),
        'trades': int((np.abs(np.diff(positions)) > 0.001).sum()),
        'maxdd_peak': str(frame['datetime'].iloc[int(np.argmax(np.maximum.accumulate(nav) - nav))].date()) if len(frame) else '',
        'maxdd_trough': str(frame['datetime'].iloc[int(np.argmin(drawdown))].date()) if len(frame) else '',
    }


def load_daily(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame['datetime'] = pd.to_datetime(frame['datetime'])
    return frame.sort_values('datetime').reset_index(drop=True)


def baseline_metrics(path: Path, label: str, window: str) -> dict:
    frame = load_daily(path)
    metrics = compute_metrics(frame[['datetime', 'price', 'position']].copy())
    return {'label': label, 'window': window, **metrics, 'path': str(path.relative_to(ROOT))}


def predict_risk_for_dates(features: pd.DataFrame, feature_cols: list[str], *, horizon: int, loss_threshold: float, probability_quantile: float, dates: pd.Series) -> pd.DataFrame:
    labelled = add_forward_drawdown_label(features, horizon, loss_threshold)
    date_series = pd.Series(pd.to_datetime(dates)).drop_duplicates().sort_values().reset_index(drop=True)
    pieces = []
    for year in sorted(date_series.dt.year.unique()):
        eval_dates = date_series[date_series.dt.year == year]
        eval_start = eval_dates.min()
        # purge train labels whose forward window would cross into evaluation time
        train_cutoff = eval_start - pd.Timedelta(days=int(horizon * 1.6) + 5)
        train = labelled[
            (labelled['datetime'] < train_cutoff)
            & (labelled[feature_cols].notna().sum(axis=1) >= 5)
            & labelled['drawdown_label'].notna()
        ].copy()
        test = labelled[labelled['datetime'].isin(set(eval_dates))].copy()
        if test.empty:
            continue
        if len(train) < 300 or train['drawdown_label'].nunique() < 2:
            risk_prob = np.zeros(len(test), dtype=float)
            threshold = 1.0
            train_rows = len(train)
            positive_rate = float(train['drawdown_label'].mean()) if len(train) else np.nan
        else:
            model = make_pipeline(
                SimpleImputer(strategy='median'),
                StandardScaler(),
                LogisticRegression(max_iter=1000, class_weight='balanced', C=0.5, random_state=0),
            )
            model.fit(train[feature_cols], train['drawdown_label'].astype(int))
            train_prob = model.predict_proba(train[feature_cols])[:, 1]
            threshold = float(np.quantile(train_prob, probability_quantile))
            risk_prob = model.predict_proba(test[feature_cols])[:, 1]
            train_rows = len(train)
            positive_rate = float(train['drawdown_label'].mean())
        part = test[['datetime']].copy()
        part['risk_prob'] = risk_prob
        part['risk_threshold'] = threshold
        part['risk_flag'] = part['risk_prob'] >= threshold
        part['train_rows'] = train_rows
        part['train_positive_rate'] = positive_rate
        pieces.append(part)
    if not pieces:
        return pd.DataFrame(columns=['datetime', 'risk_prob', 'risk_threshold', 'risk_flag', 'train_rows', 'train_positive_rate'])
    return pd.concat(pieces, ignore_index=True).sort_values('datetime')


def mix_crash_and_residual(crash_path: Path, residual_path: Path, risk: pd.DataFrame) -> pd.DataFrame:
    crash = load_daily(crash_path)
    residual = load_daily(residual_path)
    frame = crash[['datetime', 'price', 'position']].rename(columns={'position': 'crash_position'}).merge(
        residual[['datetime', 'position']].rename(columns={'position': 'residual_position'}), on='datetime', how='inner'
    )
    frame = frame.merge(risk[['datetime', 'risk_prob', 'risk_threshold', 'risk_flag', 'train_rows', 'train_positive_rate']], on='datetime', how='left')
    frame['risk_prob'] = frame['risk_prob'].fillna(0.0)
    frame['risk_threshold'] = frame['risk_threshold'].fillna(1.0)
    frame['risk_flag'] = frame['risk_flag'].fillna(False).astype(bool)
    frame['position'] = np.where(frame['risk_flag'], frame['residual_position'], frame['crash_position'])
    frame['source_path'] = np.where(frame['risk_flag'], 'residual_guard', 'crash_risk')
    frame['position_delta_vs_crash'] = frame['position'] - frame['crash_position']
    frame['position_delta_vs_residual'] = frame['position'] - frame['residual_position']
    return frame


def evaluate_candidate(features: pd.DataFrame, feature_cols: list[str], params: dict) -> tuple[dict, pd.DataFrame, list[dict]]:
    strict_dates = load_daily(STRICT_CRASH_PATH)['datetime']
    strict_risk = predict_risk_for_dates(features, feature_cols, dates=strict_dates, **params)
    strict_frame = mix_crash_and_residual(STRICT_CRASH_PATH, STRICT_RESIDUAL_PATH, strict_risk)
    strict_metrics = compute_metrics(strict_frame)
    strict_metrics.update({
        'window': 'strict_oos',
        'label': 'predictive_gate',
        'risk_days': int(strict_frame['risk_flag'].sum()),
        'risk_rate': float(strict_frame['risk_flag'].mean()),
        **params,
    })

    wf_rows = []
    for window in WF_WINDOWS:
        crash_path = WF_DIR / f'risk_budget_residual_drawdown_guard_walkforward_v1_{window}_crash_risk_soxl_daily.csv'
        residual_path = WF_DIR / f'risk_budget_residual_drawdown_guard_walkforward_v1_{window}_residual_guard_soxl_daily.csv'
        wf_dates = load_daily(crash_path)['datetime']
        wf_risk = predict_risk_for_dates(features, feature_cols, dates=wf_dates, **params)
        wf_frame = mix_crash_and_residual(crash_path, residual_path, wf_risk)
        metrics = compute_metrics(wf_frame)
        wf_rows.append({
            'window': window,
            'label': 'predictive_gate',
            **metrics,
            'risk_days': int(wf_frame['risk_flag'].sum()),
            'risk_rate': float(wf_frame['risk_flag'].mean()),
            **params,
        })
    return strict_metrics, strict_frame, wf_rows

panel = pd.read_csv(PANEL_PATH)
features, feature_cols = build_feature_frame(panel)
print('feature rows', len(features), 'feature cols', len(feature_cols), 'range', features['datetime'].min().date(), features['datetime'].max().date())


feature rows 4076 feature cols 40 range 2010-03-11 2026-05-22


In [3]:
base_rows = [
    baseline_metrics(STRICT_CRASH_PATH, 'crash_risk', 'strict_oos'),
    baseline_metrics(STRICT_RESIDUAL_PATH, 'residual_guard', 'strict_oos'),
]
for window in WF_WINDOWS:
    base_rows.append(baseline_metrics(WF_DIR / f'risk_budget_residual_drawdown_guard_walkforward_v1_{window}_crash_risk_soxl_daily.csv', 'crash_risk', window))
    base_rows.append(baseline_metrics(WF_DIR / f'risk_budget_residual_drawdown_guard_walkforward_v1_{window}_residual_guard_soxl_daily.csv', 'residual_guard', window))
base_table = pd.DataFrame(base_rows)
base_table_path = CARD_DIR / 'baseline_table.csv'
base_table.to_csv(base_table_path, index=False)
base_table[['window', 'label', 'return', 'relative_return', 'max_dd', 'avg_pos', 'trades']]


,window,label,return,relative_return,max_dd,avg_pos,trades
0,strict_oos,crash_risk,190.745831,44.599109,15.484927,0.163622,243
1,strict_oos,residual_guard,176.604683,30.457960,14.642461,0.160945,267
2,wf_2020,crash_risk,21.417928,-38.357381,22.123347,0.221486,48
3,wf_2020,residual_guard,5.580720,-54.194588,18.102212,0.172200,69
4,wf_2022,crash_risk,-9.930770,76.657302,10.970665,0.045817,8
5,wf_2022,residual_guard,-9.930770,76.657302,10.970665,0.045817,8
6,wf_2023_2024,crash_risk,40.433766,43.037195,19.079874,0.183679,40
7,wf_2023_2024,residual_guard,35.707210,38.310638,17.229977,0.159940,60


In [4]:
grid = []
for horizon in [21, 42, 63]:
    for loss_threshold in [0.10, 0.12, 0.15, 0.18, 0.22]:
        for probability_quantile in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
            grid.append({
                'horizon': horizon,
                'loss_threshold': loss_threshold,
                'probability_quantile': probability_quantile,
            })

strict_rows = []
wf_rows = []
frames_by_key = {}
for params in grid:
    strict_metrics, strict_frame, window_rows = evaluate_candidate(features, feature_cols, params)
    key = f"h{params['horizon']}_loss{params['loss_threshold']:.2f}_q{params['probability_quantile']:.2f}"
    strict_metrics['candidate_key'] = key
    strict_metrics['strict_target_pass'] = bool(strict_metrics['relative_return'] >= TARGET_RELATIVE_RETURN and strict_metrics['max_dd'] <= TARGET_MAX_DD)
    strict_rows.append(strict_metrics)
    frames_by_key[key] = strict_frame
    for row in window_rows:
        row['candidate_key'] = key
        row['wf_target_pass'] = bool(row['relative_return'] >= TARGET_RELATIVE_RETURN and row['max_dd'] <= TARGET_MAX_DD)
        wf_rows.append(row)

strict_screen = pd.DataFrame(strict_rows)
wf_screen = pd.DataFrame(wf_rows)

wf_pivot = wf_screen.pivot_table(index='candidate_key', values=['max_dd', 'relative_return', 'return'], columns='window', aggfunc='first')
wf_pivot.columns = [f'{metric}_{window}' for metric, window in wf_pivot.columns]
wf_pivot = wf_pivot.reset_index()
selection_table = strict_screen.merge(wf_pivot, on='candidate_key', how='left')
selection_table['wf_all_target_pass'] = (
    (selection_table['max_dd_wf_2020'] <= TARGET_MAX_DD)
    & (selection_table['max_dd_wf_2022'] <= TARGET_MAX_DD)
    & (selection_table['max_dd_wf_2023_2024'] <= TARGET_MAX_DD)
    & (selection_table['relative_return_wf_2022'] >= TARGET_RELATIVE_RETURN)
    & (selection_table['relative_return_wf_2023_2024'] >= TARGET_RELATIVE_RETURN)
)
selection_table['selection_score'] = (
    np.maximum(0.0, selection_table['max_dd'] - TARGET_MAX_DD) * 10.0
    + np.maximum(0.0, TARGET_RELATIVE_RETURN - selection_table['relative_return']) * 8.0
    + np.maximum(0.0, selection_table['max_dd_wf_2020'] - TARGET_MAX_DD) * 2.0
    + np.maximum(0.0, selection_table['max_dd_wf_2023_2024'] - TARGET_MAX_DD) * 2.0
    + np.maximum(0.0, -selection_table['relative_return_wf_2020']) * 0.03
    - selection_table['relative_return'] * 0.01
)
selection_table = selection_table.sort_values(['strict_target_pass', 'wf_all_target_pass', 'selection_score'], ascending=[False, False, True]).reset_index(drop=True)
selected = selection_table.iloc[0].to_dict()
selected_key = selected['candidate_key']
selected_frame = frames_by_key[selected_key].copy()
selected_frame['strategy_nav'] = 1_000_000.0 * np.r_[1.0, np.cumprod(1.0 + selected_frame['position'].to_numpy(float)[:-1] * (np.diff(selected_frame['price'].to_numpy(float)) / selected_frame['price'].to_numpy(float)[:-1]))]
selected_frame['buyhold_nav'] = 1_000_000.0 * selected_frame['price'] / selected_frame['price'].iloc[0]
selected_frame['excess'] = selected_frame['strategy_nav'] - selected_frame['buyhold_nav']
selected_frame['excess_pct'] = selected_frame['excess'] / selected_frame['buyhold_nav'] * 100.0
selected_frame['action'] = np.where(selected_frame['position'].diff().abs().fillna(0.0) > 0.001, 'TRADE', 'HOLD')

strict_screen_path = CARD_DIR / 'strict_screen.csv'
wf_screen_path = CARD_DIR / 'walkforward_screen.csv'
selection_table_path = CARD_DIR / 'selection_table.csv'
selected_daily_path = CARD_DIR / f'{CARD}_selected_soxl_daily.csv'
selection_json_path = CARD_DIR / 'selection.json'

strict_screen.to_csv(strict_screen_path, index=False)
wf_screen.to_csv(wf_screen_path, index=False)
selection_table.to_csv(selection_table_path, index=False)
selected_frame.to_csv(selected_daily_path, index=False)

selection_payload = {
    'experiment': CARD,
    'decision': 'candidate_blocked_by_walkforward',
    'selected_key': selected_key,
    'selected_params': {
        'horizon': int(selected['horizon']),
        'loss_threshold': float(selected['loss_threshold']),
        'probability_quantile': float(selected['probability_quantile']),
    },
    'strict': {k: (float(v) if isinstance(v, (np.floating, float)) and math.isfinite(float(v)) else int(v) if isinstance(v, (np.integer, int)) else bool(v) if isinstance(v, (np.bool_, bool)) else v) for k, v in selected.items() if k in ['return', 'buyhold_return', 'relative_return', 'sharpe', 'max_dd', 'avg_pos', 'trades', 'risk_days', 'risk_rate', 'strict_target_pass']},
    'walkforward': wf_screen[wf_screen['candidate_key'] == selected_key][['window', 'return', 'buyhold_return', 'relative_return', 'sharpe', 'max_dd', 'avg_pos', 'trades', 'risk_days', 'risk_rate', 'wf_target_pass']].to_dict(orient='records'),
    'baselines': base_table[['window', 'label', 'return', 'relative_return', 'max_dd', 'avg_pos', 'trades']].to_dict(orient='records'),
    'artifacts': {
        'notebook': str((CARD_DIR / f'{CARD}.ipynb').relative_to(ROOT)),
        'baseline_table': str(base_table_path.relative_to(ROOT)),
        'strict_screen': str(strict_screen_path.relative_to(ROOT)),
        'walkforward_screen': str(wf_screen_path.relative_to(ROOT)),
        'selection_table': str(selection_table_path.relative_to(ROOT)),
        'selected_daily': str(selected_daily_path.relative_to(ROOT)),
        'selection': str(selection_json_path.relative_to(ROOT)),
    },
}
selection_json_path.write_text(json.dumps(selection_payload, indent=2, sort_keys=True), encoding='utf-8')

print('grid rows', len(selection_table), 'strict target pass', int(selection_table['strict_target_pass'].sum()), 'wf all target pass', int(selection_table['wf_all_target_pass'].sum()))
print('selected', selected_key, selection_payload['selected_params'])
selection_table.head(12)[[
    'candidate_key', 'return', 'relative_return', 'max_dd', 'avg_pos', 'risk_days', 'risk_rate', 'strict_target_pass',
    'return_wf_2020', 'relative_return_wf_2020', 'max_dd_wf_2020',
    'relative_return_wf_2022', 'max_dd_wf_2022',
    'relative_return_wf_2023_2024', 'max_dd_wf_2023_2024', 'wf_all_target_pass', 'selection_score'
]]


grid rows 120 strict target pass 11 wf all target pass 0
selected h21_loss0.22_q0.20 {'horizon': 21, 'loss_threshold': 0.22, 'probability_quantile': 0.2}


,candidate_key,return,relative_return,max_dd,avg_pos,risk_days,risk_rate,strict_target_pass,return_wf_2020,relative_return_wf_2020,max_dd_wf_2020,relative_return_wf_2022,max_dd_wf_2022,relative_return_wf_2023_2024,max_dd_wf_2023_2024,wf_all_target_pass,selection_score
0,h21_loss0.22_q0.20,181.589962,35.443239,14.621835,0.161365,1076,0.685787,True,17.288341,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242,False,11.024306
1,h21_loss0.18_q0.05,179.291723,33.145001,14.642461,0.161184,1356,0.864245,True,14.818440,-44.956868,18.102212,76.657302,10.970665,38.310638,17.229977,False,11.681635
2,h21_loss0.22_q0.10,179.647616,33.500893,14.642461,0.161159,1220,0.777565,True,13.954915,-45.820393,18.102212,76.657302,10.970665,38.318647,17.229977,False,11.703981
3,h42_loss0.22_q0.05,183.314291,37.167568,14.642461,0.161443,1361,0.867431,True,4.289043,-55.486265,18.005230,76.657302,10.970665,38.310638,17.229977,False,11.763327
4,h21_loss0.22_q0.05,180.267888,34.121165,14.642461,0.161131,1364,0.869344,True,7.446592,-52.328716,18.102212,76.657302,10.970665,38.310638,17.229977,False,11.893028
5,h63_loss0.22_q0.10,185.716485,39.569762,14.702265,0.161633,1186,0.755895,True,5.580720,-54.194588,18.102212,76.657302,10.970665,38.310638,17.229977,False,11.894519
6,h63_loss0.18_q0.05,184.699461,38.552738,14.642461,0.161519,1287,0.820268,True,5.580720,-54.194588,18.102212,76.657302,10.970665,38.310638,17.229977,False,11.904689
7,h63_loss0.22_q0.05,182.665675,36.518953,14.642461,0.161450,1311,0.835564,True,5.580720,-54.194588,18.102212,76.657302,10.970665,38.310638,17.229977,False,11.925027
8,h21_loss0.18_q0.10,179.349604,33.202881,14.821542,0.161220,1261,0.803697,True,14.048431,-45.726878,20.280621,76.657302,10.970665,38.310638,17.229977,False,16.060974
9,h42_loss0.22_q0.20,183.845102,37.698379,14.821542,0.161570,1057,0.673678,True,11.053261,-48.722047,20.933222,76.657302,10.970665,38.771572,16.948846,False,16.848814


In [5]:
selected_wf = wf_screen[wf_screen['candidate_key'] == selected_key].copy()
base_comp = base_table[base_table['window'].isin(['strict_oos', *WF_WINDOWS])].copy()

readout = [
    '## Final Readout',
    '',
    'This notebook is the primary single-run conclusion artifact.',
    '',
    f"- Baseline crash-risk strict: `{base_table[(base_table['window']=='strict_oos') & (base_table['label']=='crash_risk')]['relative_return'].iloc[0]:.2f}pp` relative / `{base_table[(base_table['window']=='strict_oos') & (base_table['label']=='crash_risk')]['max_dd'].iloc[0]:.2f}%` MaxDD.",
    f"- Baseline residual-guard strict: `{base_table[(base_table['window']=='strict_oos') & (base_table['label']=='residual_guard')]['relative_return'].iloc[0]:.2f}pp` relative / `{base_table[(base_table['window']=='strict_oos') & (base_table['label']=='residual_guard')]['max_dd'].iloc[0]:.2f}%` MaxDD.",
    f"- Screen rows: `{len(selection_table)}`; strict target pass: `{int(selection_table['strict_target_pass'].sum())}`; all-window target pass: `{int(selection_table['wf_all_target_pass'].sum())}`.",
    f"- Selected key: `{selected_key}` with horizon `{int(selected['horizon'])}`, loss threshold `{float(selected['loss_threshold']):.2f}`, probability quantile `{float(selected['probability_quantile']):.2f}`.",
    f"- Selected strict: return `{selected['return']:.2f}%`, relative `{selected['relative_return']:.2f}pp`, MaxDD `{selected['max_dd']:.2f}%`, avg_pos `{selected['avg_pos']:.3f}`, risk_days `{int(selected['risk_days'])}`.",
    '',
    '### Selected Walk-Forward',
    '',
    '| window | return | relative | MaxDD | avg_pos | risk_days | target_pass |',
    '| --- | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for _, row in selected_wf.sort_values('window').iterrows():
    readout.append(
        f"| `{row['window']}` | `{row['return']:.2f}%` | `{row['relative_return']:.2f}pp` | `{row['max_dd']:.2f}%` | `{row['avg_pos']:.3f}` | `{int(row['risk_days'])}` | `{str(bool(row['wf_target_pass'])).lower()}` |"
    )
readout.extend([
    '',
    '### Decision',
    '',
    '- Diagnostic keep / no runtime promotion: predictive residual-guard release can preserve a full-window strict pass, but it does not repair saved walk-forward risk.',
    '- The selected strict path still passes `+30pp / <=15%`, but `wf_2020` remains above the 15% MaxDD guard and deeply negative versus buy-and-hold, while `wf_2023_2024` also remains above 15%.',
    '- This blocks the fixed-actor classifier overlay. The next useful loop should move the risk objective into training or actor architecture so lower peak exposure is learned before the drawdown path, rather than adding another selector over existing cash-only overlays.',
    '',
    '### Artifacts',
    '',
    f"- `{str(base_table_path.relative_to(ROOT))}`",
    f"- `{str(strict_screen_path.relative_to(ROOT))}`",
    f"- `{str(wf_screen_path.relative_to(ROOT))}`",
    f"- `{str(selection_table_path.relative_to(ROOT))}`",
    f"- `{str(selected_daily_path.relative_to(ROOT))}`",
    f"- `{str(selection_json_path.relative_to(ROOT))}`",
])
display(Markdown('\n'.join(readout)))


## Final Readout

This notebook is the primary single-run conclusion artifact.

- Baseline crash-risk strict: `44.60pp` relative / `15.48%` MaxDD.
- Baseline residual-guard strict: `30.46pp` relative / `14.64%` MaxDD.
- Screen rows: `120`; strict target pass: `11`; all-window target pass: `0`.
- Selected key: `h21_loss0.22_q0.20` with horizon `21`, loss threshold `0.22`, probability quantile `0.20`.
- Selected strict: return `181.59%`, relative `35.44pp`, MaxDD `14.62%`, avg_pos `0.161`, risk_days `1076`.

### Selected Walk-Forward

| window | return | relative | MaxDD | avg_pos | risk_days | target_pass |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| `wf_2020` | `17.29%` | `-42.49pp` | `18.53%` | `0.196` | `155` | `false` |
| `wf_2022` | `-9.93%` | `76.66pp` | `10.97%` | `0.046` | `147` | `true` |
| `wf_2023_2024` | `36.88%` | `39.49pp` | `16.52%` | `0.162` | `221` | `false` |

### Decision

- Diagnostic keep / no runtime promotion: predictive residual-guard release can preserve a full-window strict pass, but it does not repair saved walk-forward risk.
- The selected strict path still passes `+30pp / <=15%`, but `wf_2020` remains above the 15% MaxDD guard and deeply negative versus buy-and-hold, while `wf_2023_2024` also remains above 15%.
- This blocks the fixed-actor classifier overlay. The next useful loop should move the risk objective into training or actor architecture so lower peak exposure is learned before the drawdown path, rather than adding another selector over existing cash-only overlays.

### Artifacts

- `runtime/model_selection/card/predictive_drawdown_risk_gate_v1/baseline_table.csv`
- `runtime/model_selection/card/predictive_drawdown_risk_gate_v1/strict_screen.csv`
- `runtime/model_selection/card/predictive_drawdown_risk_gate_v1/walkforward_screen.csv`
- `runtime/model_selection/card/predictive_drawdown_risk_gate_v1/selection_table.csv`
- `runtime/model_selection/card/predictive_drawdown_risk_gate_v1/predictive_drawdown_risk_gate_v1_selected_soxl_daily.csv`
- `runtime/model_selection/card/predictive_drawdown_risk_gate_v1/selection.json`